In [4]:
import httpx  # install via pip install httpx
import csv
import sys
import io
from datetime import datetime

BASE_URL = "http://144.202.59.33:25503/v3"  # all endpoints use this URL base

# set params
params = {
  'symbol': 'SPXW',
  'expiration': '2025-03-19',
}

# Weekend Check (Sat/Sun)
now = datetime.now()

if now.weekday() >= 5: # 5=Sat, 6=Sun
    print("Market is Closed snapshots may not work")
    sys.exit(0)

#
# This is the streaming version, and will read line-by-line
#
url = BASE_URL + '/option/snapshot/greeks/third_order'


with httpx.stream("GET", url, params=params, timeout=60) as response:
    response.raise_for_status()  # make sure the request worked
    for line in response.iter_lines():
        for row in csv.reader(io.StringIO(line)):
            print(row)  # Now you get a parsed list of fields


HTTPStatusError: Client error '472 472' for url 'http://144.202.59.33:25503/v3/option/snapshot/greeks/third_order?symbol=SPXW&expiration=2025-03-19'
For more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/472

In [2]:
pip install httpx


[notice] A new release of pip is available: 25.1.1 -> 26.0.1
[notice] To update, run: python3.11 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [5]:
import httpx
import csv
import sys
import io
from datetime import datetime

BASE_URL = "http://144.202.59.33:25503/v3"

# Use today's date for 0DTE, or set a future expiration
now = datetime.now()

params = {
    'symbol': 'SPXW',
    'expiration': now.strftime('%Y-%m-%d'),  # today's 0DTE expiration
}

# Weekend Check (Sat/Sun)
if now.weekday() >= 5:
    print("Market is closed — snapshots may not work")
    sys.exit(0)

url = BASE_URL + '/option/snapshot/greeks/third_order'

with httpx.stream("GET", url, params=params, timeout=60) as response:
    if response.status_code != 200:
        # read whatever body the server sent back for diagnostics
        body = response.read().decode(errors="replace")
        print(f"Error {response.status_code} for {params}: {body}")
        sys.exit(1)

    header = None
    for line in response.iter_lines():
        for row in csv.reader(io.StringIO(line)):
            if header is None:
                header = row  # first row is column names
            else:
                print(dict(zip(header, row)))


{'symbol': 'SPXW', 'expiration': '2026-03-20', 'strike': '7490.000', 'right': 'PUT', 'timestamp': '2026-03-20T05:18:57.874', 'bid': '875.7000', 'ask': '899.7000', 'speed': '0.0000', 'zomma': '0.0000', 'color': '-0.0167', 'ultima': '12.4966', 'implied_vol': '1.8514', 'iv_error': '0.0000', 'underlying_timestamp': '2026-03-19T16:04:41', 'underlying_price': '6606.4900'}
{'symbol': 'SPXW', 'expiration': '2026-03-20', 'strike': '7490.000', 'right': 'CALL', 'timestamp': '2026-03-20T05:10:56.573', 'bid': '0.0000', 'ask': '0.0500', 'speed': '0.0000', 'zomma': '0.0000', 'color': '-0.0001', 'ultima': '26.5109', 'implied_vol': '1.0437', 'iv_error': '0.0304', 'underlying_timestamp': '2026-03-19T16:04:41', 'underlying_price': '6606.4900'}
{'symbol': 'SPXW', 'expiration': '2026-03-20', 'strike': '6040.000', 'right': 'PUT', 'timestamp': '2026-03-20T05:10:56.572', 'bid': '0.0000', 'ask': '0.0500', 'speed': '0.0000', 'zomma': '0.0000', 'color': '-0.0001', 'ultima': '53.0571', 'implied_vol': '0.7671', 'i

In [6]:
import httpx
import csv
import sys
import io
import os
from datetime import datetime

BASE_URL = "http://144.202.59.33:25503/v3"

now = datetime.now()

params = {
    'symbol': 'SPXW',
    'expiration': now.strftime('%Y-%m-%d'),
}

# Weekend Check (Sat/Sun)
if now.weekday() >= 5:
    print("Market is closed — snapshots may not work")
    sys.exit(0)

url = BASE_URL + '/option/snapshot/greeks/third_order'

# Ensure output directory exists
output_dir = '/workspace/testdata'
os.makedirs(output_dir, exist_ok=True)

output_file = os.path.join(output_dir, f"spxw_greeks_3rd_{now.strftime('%Y%m%d')}.csv")

with httpx.stream("GET", url, params=params, timeout=60) as response:
    if response.status_code != 200:
        body = response.read().decode(errors="replace")
        print(f"Error {response.status_code} for {params}: {body}")
        sys.exit(1)

    row_count = 0
    with open(output_file, 'w', newline='') as f:
        writer = csv.writer(f)
        for line in response.iter_lines():
            for row in csv.reader(io.StringIO(line)):
                writer.writerow(row)
                row_count += 1

    print(f"Wrote {row_count} rows (including header) to {output_file}")


Wrote 885 rows (including header) to /workspace/testdata/spxw_greeks_3rd_20260320.csv
